# Edge-IIoT Attack Trainer

Trains an Edge-IIoT binary intrusion detector (`normal` / `attack`) from a smaller Kaggle mirror of the published dataset. When enabled and `Attack_type` is available, it also trains a multi-class attack-category classifier.

This notebook uses scikit-learn Random Forest, so the Kaggle GPU setting does not accelerate the model; it can remain enabled for future GPU experiments.

In [ ]:
import csv
import random
import sys
from pathlib import Path
import json
import re

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42
# The large CSV crashes the pandas parser in Kaggle's batch runner. Use the
# standard-library CSV reader and keep balanced reservoir samples only.
CLASS_SAMPLE_SIZE = 25_000
N_ESTIMATORS = 100
TRAIN_ATTACK_TYPE = True  # Train the Edge-IIoT attack-category classifier too.
OUTPUT_DIR = Path('/kaggle/working/edge_iiot_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

all_csvs = list(Path('/kaggle/input').rglob('*.csv'))
if not all_csvs:
    raise FileNotFoundError('No CSV files were found under /kaggle/input. Check the notebook input attachment.')

def csv_score(path):
    name = path.name.lower()
    return (('ml-edgeiiot' in name or 'ml_edgeiiot' in name), 'dnn' not in name, path.stat().st_size)

data_path = max(all_csvs, key=csv_score)
print(f'Using dataset CSV: {data_path}')
def normalized_name(value):
    return re.sub(r'[^a-z0-9]', '', value.lower())

csv.field_size_limit(sys.maxsize)
rng = random.Random(RANDOM_STATE)
with data_path.open('r', encoding='utf-8', errors='replace', newline='') as handle:
    reader = csv.reader(handle)
    headers = next(reader)
    header_lookup = {normalized_name(header): index for index, header in enumerate(headers)}
    label_index = next((header_lookup[name] for name in ('attacklabel', 'label') if name in header_lookup), None)
    if label_index is None:
        raise ValueError(f'No attack label column was found in CSV header: {headers}')

    reservoirs = {'benign': [], 'attack': []}
    seen = {'benign': 0, 'attack': 0}
    for row_number, row in enumerate(reader, start=1):
        if len(row) != len(headers):
            continue
        raw_label = row[label_index].strip().lower()
        group = 'benign' if raw_label in {'normal', 'benign', '0', 'false', 'no'} else 'attack'
        seen[group] += 1
        reservoir = reservoirs[group]
        if len(reservoir) < CLASS_SAMPLE_SIZE:
            reservoir.append(row)
        else:
            replacement_index = rng.randrange(seen[group])
            if replacement_index < CLASS_SAMPLE_SIZE:
                reservoir[replacement_index] = row
        if row_number % 250_000 == 0:
            print(f'Scanned {row_number:,} rows; class counts seen: {seen}')

if not reservoirs['benign'] or not reservoirs['attack']:
    raise ValueError(f'Could not obtain both classes from Edge-IIoT. Rows seen: {seen}')
df = pd.DataFrame(reservoirs['benign'] + reservoirs['attack'], columns=headers)
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
print(f'Sampled rows: {len(df):,}; columns: {len(df.columns)}; source rows seen: {seen}')
print(f'Sampled rows: {len(df):,}; columns: {len(df.columns)}')
display(df.head())

In [ ]:
def find_column(candidates):
    by_normalized = {re.sub(r'[^a-z0-9]', '', c.lower()): c for c in df.columns}
    for candidate in candidates:
        match = by_normalized.get(re.sub(r'[^a-z0-9]', '', candidate.lower()))
        if match:
            return match
    return None

binary_target = find_column(['Attack_label', 'attack_label', 'Label', 'label'])
attack_type_target = find_column(['Attack_type', 'attack_type', 'Attack Type', 'attack type'])
if binary_target is None:
    raise ValueError(f'Could not find the binary label column. Available columns: {list(df.columns)}')

# Remove both label columns from features to prevent label leakage.
label_columns = [c for c in [binary_target, attack_type_target] if c]
X = df.drop(columns=label_columns).copy()
X = X.replace([np.inf, -np.inf], np.nan)

# IDs are capture-specific and tend to create a model that does not generalize.
id_like = [c for c in X.columns if re.search(r'(^id$|^unnamed|flow.?id|timestamp)', c, re.IGNORECASE)]
X = X.drop(columns=id_like, errors='ignore')

y_binary_raw = df[binary_target].astype(str).str.strip().str.lower()
benign_values = {'normal', 'benign', '0', 'false', 'no'}
y_binary = (~y_binary_raw.isin(benign_values)).astype(int)
if y_binary.nunique() < 2:
    raise ValueError(f'Binary target {binary_target!r} has fewer than two classes: {y_binary_raw.value_counts().to_dict()}')

numeric_columns = X.select_dtypes(include=[np.number, 'bool']).columns.tolist()
categorical_columns = [c for c in X.columns if c not in numeric_columns]
print(f'Binary target: {binary_target}; attack-type target: {attack_type_target}')
print(f'Features: {len(X.columns)} ({len(numeric_columns)} numeric, {len(categorical_columns)} categorical); dropped IDs: {id_like}')
print('Binary class counts:', y_binary.value_counts().sort_index().to_dict())

In [ ]:
def build_pipeline():
    transformers = []
    if numeric_columns:
        transformers.append(('numeric', SimpleImputer(strategy='median'), numeric_columns))
    if categorical_columns:
        categorical_pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ])
        transformers.append(('categorical', categorical_pipeline, categorical_columns))
    return Pipeline([
        ('preprocess', ColumnTransformer(transformers=transformers, remainder='drop')),
        ('classifier', RandomForestClassifier(
            n_estimators=N_ESTIMATORS,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight='balanced_subsample'
        ))
    ])

def json_safe(value):
    if pd.isna(value):
        return None
    if isinstance(value, np.generic):
        return value.item()
    return value

def held_out_demo_samples(test_x, test_y, predictions, limit=5):
    prediction_by_index = dict(zip(test_x.index, predictions))
    samples = []
    for actual_label in pd.unique(test_y):
        candidates = test_y[test_y == actual_label].index.tolist()
        index = next((item for item in candidates if prediction_by_index[item] == actual_label), candidates[0])
        samples.append({
            'expected_label': json_safe(test_y.loc[index]),
            'predicted_label': json_safe(prediction_by_index[index]),
            'features': {name: json_safe(value) for name, value in test_x.loc[index].items()}
        })
        if len(samples) == limit:
            break
    return samples

def train_and_save(name, y):
    task_x = X.loc[y.index]
    train_x, test_x, train_y, test_y = train_test_split(
        task_x, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
    )
    pipeline = build_pipeline()
    pipeline.fit(train_x, train_y)
    predictions = pipeline.predict(test_x)
    metrics = {
        'dataset': 'Edge-IIoT',
        'task': name,
        'rows': int(len(task_x)),
        'raw_feature_count': int(len(X.columns)),
        'accuracy': float(accuracy_score(test_y, predictions)),
        'macro_f1': float(f1_score(test_y, predictions, average='macro')),
        'class_counts': {str(k): int(v) for k, v in y.value_counts().sort_index().items()},
        'classification_report': classification_report(test_y, predictions, output_dict=True, zero_division=0),
        'data_file': data_path.name
    }
    joblib.dump(pipeline, OUTPUT_DIR / f'edge_iiot_{name}_model.joblib')
    (OUTPUT_DIR / f'edge_iiot_{name}_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    print(f'\n{name} model saved. Accuracy={metrics["accuracy"]:.5f}; macro F1={metrics["macro_f1"]:.5f}')
    return metrics, held_out_demo_samples(test_x, test_y, predictions)

binary_metrics, binary_demo_samples = train_and_save('binary', y_binary)
binary_metrics

In [ ]:
if TRAIN_ATTACK_TYPE and attack_type_target:
    y_attack_type = df[attack_type_target].astype(str).str.strip()
    valid = y_attack_type.notna() & (y_attack_type != '')
    if y_attack_type[valid].nunique() > 1:
        multiclass_metrics, attack_type_demo_samples = train_and_save('attack_type', y_attack_type[valid])
    else:
        multiclass_metrics = {'skipped': 'Attack_type contains fewer than two classes.'}
        attack_type_demo_samples = []
elif not TRAIN_ATTACK_TYPE:
    multiclass_metrics = {'skipped': 'Disabled by configuration.'}
    attack_type_demo_samples = []
else:
    multiclass_metrics = {'skipped': 'No Attack_type column found.'}
    attack_type_demo_samples = []

summary = {'binary': binary_metrics, 'attack_type': multiclass_metrics}
(OUTPUT_DIR / 'training_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
demo_samples = {
    'dataset': 'Edge-IIoT',
    'source': 'Correctly classified held-out test records from the Kaggle training split.',
    'binary': binary_demo_samples,
    'attack_type': attack_type_demo_samples
}
(OUTPUT_DIR / 'edge_iiot_demo_samples.json').write_text(json.dumps(demo_samples, indent=2), encoding='utf-8')
print('\nOutput files:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f' - {path.name} ({path.stat().st_size / 1024 / 1024:.2f} MB)')